In [1]:
from pathlib import Path
# from transit_area_buffer_utils import station_buffer, logger_process
import geopandas as gpd
import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors
import folium
from folium.plugins import DualMap
# import ipywidgets

### Data request:
How many new housing units (affordable and market-rate) were planned for around rail stations in PBA 2050+.

### Methodology:
The following calculation applies the methodology from PBA 2050+ Performance Metrics:
* 2023 condition is based on interpolating parcel-level BAUS output of PBA50+ NoProject model run ("PBA50Plus_NoProject_v38")
* 2050 condition is based on parcel-level BAUS output of PBA50+ Final Blueprint model run ("PBA50Plus_Final_Blueprint_v65")
* to tally units "around rail stations", first, a set of 2023 rail station points and a set of 2050 rail station points were created, then a 1/2 mile buffer area was created around each station, then parcels that are within the buffer areas were selected, lastly, units on these parcels were considered "within" rail station buffers, in other words, "around rail stations". Therefore, the calculated "new/growth ... around rail stations" reflects both an expansion of the "rail station area" and the growth itself
* the data request says "housing units (affordable and market-rate)", but this script summarized housing units, affordable units (deed-restricted units), total households, low-income (Q1) households, and total employment in case these metrics are useful

In [2]:
ANALYSIS_CRS = 'EPSG:26910'  # California Albers

### Load data and create crosswalks: rail station buffers spatial data, p10 parcel spatial data

In [3]:
# load rail station buffer shapefiles

# Box folder path
BOX_dir = Path('E:/Box/Modeling and Surveys/Urban Modeling')

working_dir = BOX_dir / 'Spatial/transit/transit_service_levels/update_2025/bespoke_railStations'
stations_2023_file = working_dir / 'Rail_stations_2023_halfmilebuffers' / 'Rail_stations_2023_halfmilebuffers.shp'
stations_2023_gdf = gpd.read_file(stations_2023_file)
stations_2050_file = working_dir / 'Rail_stations_2050_halfmilebuffers' / 'Rail_stations_2050_halfmilebuffers.shp'
stations_2050_gdf = gpd.read_file(stations_2050_file)

In [4]:
# take a look at what's in the data
display(stations_2023_gdf)
display(stations_2050_gdf)

,Shape_Leng,Shape_Area,geometry
0,6.509236,0.034958,"MULTIPOLYGON (((-121.56594 36.99624, -121.5662..."


,Shape_Leng,Shape_Area,geometry
0,682342.041055,3.536809e+08,"MULTIPOLYGON (((627607.057 4095416.012, 627581..."


In [5]:
center = stations_2023_gdf.to_crs(epsg=4326).geometry.centroid
center_lat = center.y.mean()
center_lon = center.x.mean()

m = DualMap(location=[center_lat, center_lon], zoom_start=10, layout='horizontal')

# Folium tooltips cannot use the geometry field; keep only attribute fields.
attrs_2023 = [c for c in stations_2023_gdf.columns if c != 'geometry'][:3]
attrs_2050 = [c for c in stations_2050_gdf.columns if c != 'geometry'][:3]
tooltip_2023 = folium.GeoJsonTooltip(fields=attrs_2023) if attrs_2023 else None
tooltip_2050 = folium.GeoJsonTooltip(fields=attrs_2050) if attrs_2050 else None

folium.GeoJson(
    stations_2023_gdf.to_crs(epsg=4326),
    name='2023 Rail Stations (0.5mi buffer)',
    style_function=lambda _: {
        'fillColor': 'steelblue', 'color': 'navy',
        'weight': 1, 'fillOpacity': 0.5
    },
    tooltip=tooltip_2023
).add_to(m.m1)
folium.map.Marker(
    [center_lat + 0.01, center_lon],
    icon=folium.DivIcon(html='<b style="font-size:14px;color:navy">2023</b>')
).add_to(m.m1)

folium.GeoJson(
    stations_2050_gdf.to_crs(epsg=4326),
    name='2050 Rail Stations (0.5mi buffer)',
    style_function=lambda _: {
        'fillColor': 'tomato', 'color': 'darkred',
        'weight': 1, 'fillOpacity': 0.4
    },
    tooltip=tooltip_2050
).add_to(m.m2)
folium.map.Marker(
    [center_lat + 0.01, center_lon],
    icon=folium.DivIcon(html='<b style="font-size:14px;color:darkred">2050</b>')
).add_to(m.m2)

folium.LayerControl().add_to(m)
m.save(working_dir / "rail_stations.html")
m

C:\Users\ywang\AppData\Local\Temp\ipykernel_7408\2543371070.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = stations_2023_gdf.to_crs(epsg=4326).geometry.centroid


In [6]:
# load p10 parcels shapefiles

parcels_geo_file = BOX_dir / 'Bay Area UrbanSim/PBA50/Policies/Base zoning/inputs/p10_geo.feather'
parcels_geo = gpd.read_feather(parcels_geo_file)

print(len(parcels_geo), 'parcels loaded')
display(parcels_geo.head())

1956208 parcels loaded


,PARCEL_ID,ACRES,COUNTY_ID,ZONE_ID,APN,GEOM_ID,ID,CENTROID,X,Y,geom_id_s,Shape_Leng,Shape_Area,geometry
0,229116.0,3.360520,1.0,725.0,099 029001700,1.030511e+13,14224.0,0101000020D00A00000FDDF39AF3C53C41889F50423032...,-121.795620,37.655379,10305106092872,1682.739489,1.359435e+04,"POLYGON ((605993.404 4168316.032, 605993.297 4..."
1,244166.0,1.294423,1.0,715.0,099B540210200,1.110735e+13,16362.0,0101000020D00A000081F795B4C5E23C419ACAA6DA1667...,-121.713004,37.717277,11107351665227,701.031207,5.236558e+03,"POLYGON ((613491.065 4175262.642, 613490.995 4..."
2,202378.0,14.993605,1.0,820.0,085A643106000,1.103018e+13,16900.0,0101000020D00A0000BA9166219F7A3C411C7B5DC25834...,-122.014199,37.655260,11030175960628,6044.820897,6.064821e+04,"POLYGON ((586745.708 4168287.089, 586746.547 4..."
3,2004420.0,316.247146,97.0,1401.0,141-100-012,6.381678e+12,19308.0,0101000020D00A00001ED46C59D1803B4134422EA6C9E0...,-122.771868,38.727893,6381677629073,4935.308767,1.278606e+06,"POLYGON ((520226.869 4286274.188, 520222.958 4..."
4,340332.0,0.621275,1.0,763.0,525 166004800,3.148755e+11,25266.0,0101000020D00A0000AF541F4B8E873C418111D5C36DD5...,-121.974508,37.546277,314875459798,833.808716,2.513031e+03,"POLYGON ((590655.279 4155999.718, 590590.988 4..."


In [7]:
# create a p10 parcel - station buffer crosswalk

# first, convert to CRS 26910
stations_2023_gdf = stations_2023_gdf.to_crs(ANALYSIS_CRS)
stations_2050_gdf = stations_2050_gdf.to_crs(ANALYSIS_CRS)
parcels_geo = parcels_geo.to_crs(ANALYSIS_CRS)

# spatial join - use same joining methodology as the PBA50+ performance metrics, which was a "within" join (parcels within station buffers)
# https://github.com/BayAreaMetro/bayarea_urbansim/blob/fbp_mod_updates/scripts/metrics/create_transit_service_areas2.py#L62-L63
parcels_within_buffer_2023 = gpd.sjoin(parcels_geo, stations_2023_gdf, predicate="within", how="left")
parcels_within_buffer_2023['railStations_2023'] = parcels_within_buffer_2023.index_right.notnull().astype(int)

parcels_within_buffer_2050 = gpd.sjoin(parcels_geo, stations_2050_gdf, predicate="within", how="left")
parcels_within_buffer_2050['railStations_2050'] = parcels_within_buffer_2050.index_right.notnull().astype(int)

# merge into one dataframe
parcels_within_railStation_buffers = parcels_within_buffer_2023[['PARCEL_ID', 'railStations_2023']].merge(
    parcels_within_buffer_2050[['PARCEL_ID', 'railStations_2050']],
    on='PARCEL_ID',
    how='outer'
)
print(len(parcels_within_railStation_buffers), 'parcels in crosswalk')

# basic sanity check
print(parcels_within_buffer_2023['railStations_2023'].sum(), 'parcels within 2023 rail station buffers')
print(parcels_within_buffer_2050['railStations_2050'].sum(), 'parcels within 2050 rail station buffers')

1956208 parcels in crosswalk
229992 parcels within 2023 rail station buffers
236169 parcels within 2050 rail station buffers


In [8]:
# also just check the parcel crosswalks we had in PBA50+ performance metrics
# those were based on HighQuality transit areas, which include non-rail stops, so should have more parcels than the rail station buffers.

parcels_HQ_transit_file = BOX_dir / 'Spatial/transit/transit_service_levels/update_2025/outputs/parcels10_x_high_quality_stop_buffer.csv'
parcels_HQ_transit = pd.read_csv(parcels_HQ_transit_file)

# take a look at the data
display(parcels_HQ_transit.head())

# "cur" represents 2023 conditions; "fbp" represents 2050 conditions
print(parcels_HQ_transit['cur'].sum(), 'parcels within 2023 high-quality transit buffers')
print(parcels_HQ_transit['fbp'].sum(), 'parcels within 2050 high-quality transit buffers')


,parcel_id,cur,np,dbp,fbp
0,229116.0,0,0,0,0
1,244166.0,0,0,0,0
2,202378.0,0,0,0,0
3,2004420.0,0,0,0,0
4,340332.0,0,0,0,1


596129 parcels within 2023 high-quality transit buffers
824877 parcels within 2050 high-quality transit buffers


### BAUS parcel output from PBA50+

In [9]:
# BAUS model output
output_dir = Path(r"M:\urban_modeling\baus\PBA50Plus")
                  
# NoProject data - for 2023 interpolation
p2020_file = output_dir / 'PBA50Plus_NoProject/PBA50Plus_NoProject_v38/core_summaries/PBA50Plus_NoProject_v38_parcel_summary_2020.csv'
p2025_file = output_dir / 'PBA50Plus_NoProject/PBA50Plus_NoProject_v38/core_summaries/PBA50Plus_NoProject_v38_parcel_summary_2025.csv'
p2020 = pd.read_csv(p2020_file)
p2025 = pd.read_csv(p2025_file)

# FBP data - for 2050

p2050_file = output_dir / 'PBA50Plus_FinalBlueprint/PBA50Plus_Final_Blueprint_v65/core_summaries/PBA50Plus_Final_Blueprint_v65_parcel_summary_2050.csv'
p2050 = pd.read_csv(p2050_file)
p2050 = p2050[['parcel_id', 'tothh', 'hhq1', 'totemp', 'deed_restricted_units', 'residential_units']].rename(
    columns={
        'tothh': 'tothh_2050',
        'hhq1': 'hhq1_2050',
        'totemp': 'totemp_2050',
        'deed_restricted_units': 'deed_restricted_units_2050',
        'residential_units': 'residential_units_2050'})

In [10]:
# interpolate 2023 data from 2020 and 2025
p2020_2025 = p2020[['parcel_id', 'tothh', 'hhq1', 'totemp', 'deed_restricted_units', 'residential_units']].merge(
    p2025[['parcel_id', 'tothh', 'hhq1', 'totemp', 'deed_restricted_units', 'residential_units']], on='parcel_id', suffixes=('_2020', '_2025'), indicator=True)
print(p2020_2025['_merge'].value_counts())

for col in ['tothh', 'totemp', 'hhq1', 'deed_restricted_units', 'residential_units']:
    p2020_2025[col + '_2020'] = p2020_2025[col + '_2020'].fillna(0)
    p2020_2025[col + '_2025'] = p2020_2025[col + '_2025'].fillna(0)
    p2020_2025[col + '_2023'] = p2020_2025[col + '_2020'] + (p2020_2025[col + '_2025'] - p2020_2025[col + '_2020']) * 3 / 5

display(p2020_2025.head())

_merge
both          1956212
left_only           0
right_only          0
Name: count, dtype: int64


,parcel_id,tothh_2020,hhq1_2020,totemp_2020,deed_restricted_units_2020,residential_units_2020,tothh_2025,hhq1_2025,totemp_2025,deed_restricted_units_2025,residential_units_2025,_merge,tothh_2023,totemp_2023,hhq1_2023,deed_restricted_units_2023,residential_units_2023
0,229116,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,both,0.0,0.0,0.0,0.0,0.0
1,244166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,both,0.0,0.0,0.0,0.0,0.0
2,202378,17.0,2.0,0.0,0.0,17.0,17.0,2.0,0.0,0.0,17.0,both,17.0,0.0,2.0,0.0,17.0
3,2004420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,both,0.0,0.0,0.0,0.0,0.0
4,340332,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,both,0.0,0.0,0.0,0.0,0.0


### merge and summarize

In [11]:
parcel_growth = p2020_2025.loc[:, p2020_2025.columns != '_merge'].merge(
    p2050,
    on='parcel_id',
    how='outer'
)
print(len(parcel_growth), 'parcels in merged parcel table')

parcel_growth = parcel_growth.merge(
    parcels_within_railStation_buffers.rename(columns={'PARCEL_ID': 'parcel_id'}),
    on='parcel_id',
    how='outer',
    indicator=True
)
print(len(parcel_growth), 'parcels in merged parcel table after merging with rail station buffer crosswalk')
print(parcel_growth['_merge'].value_counts())

# consider the 4 synthetic parcels not within rail station buffers 
parcel_growth['railStations_2023'] = parcel_growth['railStations_2023'].fillna(0).astype(int)
parcel_growth['railStations_2050'] = parcel_growth['railStations_2050'].fillna(0).astype(int)

1956212 parcels in merged parcel table
1956212 parcels in merged parcel table after merging with rail station buffer crosswalk
_merge
both          1956208
left_only           4
right_only          0
Name: count, dtype: int64


In [12]:
# summarize households and employment by rail-station buffer flags

summary_2023 = (
    parcel_growth
    .groupby('railStations_2023', dropna=False)[['tothh_2023', 'hhq1_2023', 'totemp_2023', 'deed_restricted_units_2023', 'residential_units_2023']]
    .sum()
    .reset_index()
)
summary_2023_total = pd.DataFrame([
    {
        'railStations_2023': 'total',
        'tothh_2023': summary_2023['tothh_2023'].sum(),
        'hhq1_2023': summary_2023['hhq1_2023'].sum(),
        'totemp_2023': summary_2023['totemp_2023'].sum(),
        'deed_restricted_units_2023': summary_2023['deed_restricted_units_2023'].sum(),
        'residential_units_2023': summary_2023['residential_units_2023'].sum()
    }
])
summary_2023 = pd.concat([summary_2023, summary_2023_total], ignore_index=True)

summary_2050 = (
    parcel_growth
    .groupby('railStations_2050', dropna=False)[['tothh_2050', 'hhq1_2050', 'totemp_2050', 'deed_restricted_units_2050', 'residential_units_2050']]
    .sum()
    .reset_index()
)
summary_2050_total = pd.DataFrame([
    {
        'railStations_2050': 'total',
        'tothh_2050': summary_2050['tothh_2050'].sum(),
        'hhq1_2050': summary_2050['hhq1_2050'].sum(),
        'totemp_2050': summary_2050['totemp_2050'].sum(),
        'deed_restricted_units_2050': summary_2050['deed_restricted_units_2050'].sum(),
        'residential_units_2050': summary_2050['residential_units_2050'].sum()
    }
])
summary_2050 = pd.concat([summary_2050, summary_2050_total], ignore_index=True)

print('2023 totals grouped by railStations_2023')
display(summary_2023)

print('2050 totals grouped by railStations_2050')
display(summary_2050)

2023 totals grouped by railStations_2023


,railStations_2023,tothh_2023,hhq1_2023,totemp_2023,deed_restricted_units_2023,residential_units_2023
0,0,2286769.4,515973.0,2791727.8,85073.0,2450767.4
1,1,571911.2,193962.2,1310736.6,43898.4,616463.2
2,total,2858680.6,709935.2,4102464.4,128971.4,3067230.6


2050 totals grouped by railStations_2050


,railStations_2050,tothh_2050,hhq1_2050,totemp_2050,deed_restricted_units_2050,residential_units_2050
0,0,2742353.0,524751.0,3652337.0,537872.0,2901040.0
1,1,1053479.0,418727.0,1783282.0,550260.0,1089911.0
2,total,3795832.0,943478.0,5435619.0,1088132.0,3990951.0


In [13]:
# relabel 0/1/total, set index, then merge 2023 and 2050 summaries
label_map = {1: 'within', 0: 'outside', 'total': 'total'}

summary_2023_labeled = summary_2023.copy()
summary_2023_labeled['buffer_group'] = summary_2023_labeled['railStations_2023'].map(label_map)
summary_2023_labeled = summary_2023_labeled.drop(columns=['railStations_2023']).set_index('buffer_group')

summary_2050_labeled = summary_2050.copy()
summary_2050_labeled['buffer_group'] = summary_2050_labeled['railStations_2050'].map(label_map)
summary_2050_labeled = summary_2050_labeled.drop(columns=['railStations_2050']).set_index('buffer_group')

summary_merged = summary_2023_labeled.merge(
    summary_2050_labeled,
    left_index=True,
    right_index=True,
    how='outer'
)

summary_merged = summary_merged.reindex(['within', 'outside', 'total'])

display(summary_merged)

,tothh_2023,hhq1_2023,totemp_2023,deed_restricted_units_2023,residential_units_2023,tothh_2050,hhq1_2050,totemp_2050,deed_restricted_units_2050,residential_units_2050
buffer_group,,,,,,,,,,
within,571911.2,193962.2,1310736.6,43898.4,616463.2,1053479.0,418727.0,1783282.0,550260.0,1089911.0
outside,2286769.4,515973.0,2791727.8,85073.0,2450767.4,2742353.0,524751.0,3652337.0,537872.0,2901040.0
total,2858680.6,709935.2,4102464.4,128971.4,3067230.6,3795832.0,943478.0,5435619.0,1088132.0,3990951.0


In [14]:
# calculate growth
for col in ['tothh', 'hhq1', 'totemp', 'deed_restricted_units', 'residential_units']:
    summary_merged[col + '_growth'] = summary_merged[col + '_2050'] - summary_merged[col + '_2023']

summary_merged = summary_merged.reset_index()
display(summary_merged)
summary_merged.to_csv(working_dir / 'rail_station_buffer_summary.csv', index=False)

,buffer_group,tothh_2023,hhq1_2023,totemp_2023,deed_restricted_units_2023,residential_units_2023,tothh_2050,hhq1_2050,totemp_2050,deed_restricted_units_2050,residential_units_2050,tothh_growth,hhq1_growth,totemp_growth,deed_restricted_units_growth,residential_units_growth
0,within,571911.2,193962.2,1310736.6,43898.4,616463.2,1053479.0,418727.0,1783282.0,550260.0,1089911.0,481567.8,224764.8,472545.4,506361.6,473447.8
1,outside,2286769.4,515973.0,2791727.8,85073.0,2450767.4,2742353.0,524751.0,3652337.0,537872.0,2901040.0,455583.6,8778.0,860609.2,452799.0,450272.6
2,total,2858680.6,709935.2,4102464.4,128971.4,3067230.6,3795832.0,943478.0,5435619.0,1088132.0,3990951.0,937151.4,233542.8,1333154.6,959160.6,923720.4
